In [1]:
# ============================================================
# 📦 INSTALL ALL REQUIRED LIBRARIES (RUN ONCE)
# ============================================================

# Core scientific stack
!pip install -q numpy pandas scipy scikit-learn matplotlib seaborn

# Radiomics
!pip install -q SimpleITK nibabel
!pip install git+https://github.com/AIM-Harvard/pyradiomics.git

# Feature selection
!pip install -q pymrmr
!pip install -q deap

# Utility
!pip install -q openpyxl tqdm


  Cloning https://github.com/AIM-Harvard/pyradiomics.git to /tmp/pip-req-build-w3bcvd9v
  Running command git clone --filter=blob:none --quiet https://github.com/AIM-Harvard/pyradiomics.git /tmp/pip-req-build-w3bcvd9v
  Resolved https://github.com/AIM-Harvard/pyradiomics.git to commit 8ed579383b44806651c463d5e691f3b2b57522ab
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for pyradiomics: filename=pyradiomics-3.1.1.dev111+g8ed579383-cp312-cp312-linux_x86_64.whl size=121811 sha256=a05350566b03a2375a877310ffffcc495ec032eea8fceed8b950a000b479ec67
  Stored in directory: /tmp/pip-ephem-wheel-cache-fblyx8zi/wheels/e2/99/a6/b2d774920a9e48d28ec790aa277aa8f49f531d2d6d63cd5c94
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=bc7fc991b7c9c84cd55aa4c4a0a20001c89dbe4f74207489

In [2]:
# ============================================================
# SEL 1 — STANDARDIZATION (RESAMPLE SPACING ONLY, SAFE FOR CT+MRI)
# Output: standardized_images/<Patient>/<Protocol>/
#         image_std.nii.gz + mask_std.nii.gz
# ============================================================

import os
import SimpleITK as sitk

BASE_DIR = "/kaggle/input/nifti-renal-cancer-clear-cell/NIFTI - Renal Cancer Clear Cell"
IMAGE_DIR = os.path.join(BASE_DIR, "Image")
MASK_DIR  = os.path.join(BASE_DIR, "Mask")

STD_DIR = "/kaggle/working/standardized_images"
os.makedirs(STD_DIR, exist_ok=True)

def resample_isotropic(img, spacing=1.0, interpolator=sitk.sitkBSpline):
    original_spacing = img.GetSpacing()
    original_size    = img.GetSize()

    new_size = [
        int(round(original_size[i] * (original_spacing[i] / spacing)))
        for i in range(3)
    ]

    resampler = sitk.ResampleImageFilter()
    resampler.SetInterpolator(interpolator)
    resampler.SetOutputSpacing((spacing, spacing, spacing))
    resampler.SetSize(new_size)
    resampler.SetOutputOrigin(img.GetOrigin())
    resampler.SetOutputDirection(img.GetDirection())
    resampler.SetDefaultPixelValue(0)

    return resampler.Execute(img)


# ============================================================
# RUN STANDARDIZATION
# ============================================================

for patient_id in sorted(os.listdir(IMAGE_DIR)):
    patient_img_dir = os.path.join(IMAGE_DIR, patient_id)
    patient_mask_dir = os.path.join(MASK_DIR, patient_id)

    if not os.path.isdir(patient_img_dir):
        continue

    for protocol in sorted(os.listdir(patient_img_dir)):
        protocol_img_dir = os.path.join(patient_img_dir, protocol)
        protocol_mask_dir = os.path.join(patient_mask_dir, protocol)

        if not os.path.isdir(protocol_img_dir):
            continue

        out_dir = os.path.join(STD_DIR, patient_id, protocol)
        os.makedirs(out_dir, exist_ok=True)

        for img_file in sorted(os.listdir(protocol_img_dir)):
            if not img_file.endswith((".nii", ".nii.gz")):
                continue

            ext = ".nii.gz" if img_file.endswith(".nii.gz") else ".nii"
            base = img_file.replace(ext, "")
            mask_file = f"{base.replace('-', '')} segmentation{ext}"

            img_path = os.path.join(protocol_img_dir, img_file)
            mask_path = os.path.join(protocol_mask_dir, mask_file)

            if not os.path.exists(mask_path):
                print(f"❌ Mask not found for: {img_file}")
                continue

            print(f"🧪 Standardizing → {patient_id}/{protocol}/{img_file}")

            try:
                img = sitk.ReadImage(img_path)
                msk = sitk.ReadImage(mask_path)

                img_std = resample_isotropic(img, spacing=1, interpolator=sitk.sitkBSpline)
                msk_std = resample_isotropic(msk, spacing=1, interpolator=sitk.sitkNearestNeighbor)

                sitk.WriteImage(img_std, os.path.join(out_dir, f"{base}_std.nii.gz"))
                sitk.WriteImage(msk_std, os.path.join(out_dir, f"{base}_std_mask.nii.gz"))

            except Exception as e:
                print("❌ Error:", e)

print("\n======================================")
print(" STANDARDIZATION COMPLETED")
print("======================================")
print("Saved to:", STD_DIR)


🧪 Standardizing → TCGA-B0-4712/12-09-1991-NA-CT ABDOMEN WITH CONTRA-96104/2 HELICAL 5s_2.nii
🧪 Standardizing → TCGA-B0-4821/10-28-1984-NA-ABDOMEN WWO-67734/3 Unnamed Series - acquisitionNumber 2.nii
🧪 Standardizing → TCGA-B0-5085/06-18-1987-NA-e1 AP-18854/3 POST CONTRAST.nii
🧪 Standardizing → TCGA-B0-5099/12-29-1984-NA-CT ABDOMEN-49423/3 AXIAL SCANS.nii
🧪 Standardizing → TCGA-B0-5106/04-11-1987-NA-NA-14123/2 RENALS-T1_AXIAL_BREATHHOLD.nii
🧪 Standardizing → TCGA-B0-5115/06-17-1992-NA-AP-82538/3 NEPHROGRAPHIC PHASE.nii
🧪 Standardizing → TCGA-B0-5706/03-08-1987-NA-CT CA P WO WITH-13559/3 Unnamed Series - acquisitionNumber 2.nii
🧪 Standardizing → TCGA-B0-5712/03-05-1987-NA-KIDNEYS-72228/7 ax dynamic c.nii
🧪 Standardizing → TCGA-B8-4143/10-13-1995-NA-MRI ABDOMEN PELVIS WWO CONT-06267/9 LIVER-KIDNEY-TI_FL2D_AXIAL.nii
🧪 Standardizing → TCGA-B8-4151/05-19-2003-NA-CT RENAL Mass-57015/5 Venous Phase  5.0  B30f.nii
🧪 Standardizing → TCGA-B8-5158/12-28-2003-NA-CT RENAL Mass-54722/4 Arterial  5.0  

In [3]:
# ============================================================
# SEL 2 — PYRADIOMICS USING STANDARDIZED IMAGES
# ============================================================

import os
import pandas as pd
from radiomics import featureextractor

STD_DIR = "/kaggle/working/standardized_images"
OUTPUT_DIR = "/kaggle/working/radiomics_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_CSV = os.path.join(OUTPUT_DIR, "radiomics_features.csv")

# ============================================================
# PYRADIOMICS CONFIGURATION
# ============================================================

params = {
    "binWidth": 25,
    "interpolator": "sitkBSpline",
    "resampledPixelSpacing": [1, 1, 1],     # already standardized
    "enableCExtensions": True,
    "geometryTolerance": 2,
    "correctMask": True,
    "label": 1
}

extractor = featureextractor.RadiomicsFeatureExtractor(**params)

# Enable feature classes
extractor.enableAllFeatures()

# Enable high-order image types
extractor.enableImageTypeByName("Original")
extractor.enableImageTypeByName("Wavelet")
extractor.enableImageTypeByName("LoG", customArgs={"sigma": [1.0, 2.0, 3.0]})
extractor.enableImageTypeByName("Square")
extractor.enableImageTypeByName("SquareRoot")
extractor.enableImageTypeByName("Logarithm")
extractor.enableImageTypeByName("Exponential")
extractor.enableImageTypeByName("Gradient")

# ============================================================
# FEATURE EXTRACTION LOOP
# ============================================================

all_features = []

for patient_id in sorted(os.listdir(STD_DIR)):
    patient_dir = os.path.join(STD_DIR, patient_id)

    for protocol in sorted(os.listdir(patient_dir)):
        protocol_dir = os.path.join(patient_dir, protocol)

        for file in sorted(os.listdir(protocol_dir)):
            if not file.endswith("_std.nii.gz"):
                continue

            img_path = os.path.join(protocol_dir, file)
            mask_path = img_path.replace("_std.nii.gz", "_std_mask.nii.gz")

            if not os.path.exists(mask_path):
                print("❌ Missing mask:", mask_path)
                continue

            print(f"🔍 Extracting → {patient_id}/{protocol}/{file}")

            try:
                result = extractor.execute(img_path, mask_path)

                row = {
                    "patient_id": patient_id,
                    "protocol": protocol,
                    "image_file": file,
                }

                for k, v in result.items():
                    if not k.startswith("diagnostics"):
                        row[k] = v

                all_features.append(row)

            except Exception as e:
                print("❌ Radiomics error:", e)

# ============================================================
# SAVE RESULTS
# ============================================================

df = pd.DataFrame(all_features)
df.to_csv(OUTPUT_CSV, index=False)

print("\n======================================")
print(" RADIOMICS EXTRACTION COMPLETED")
print("======================================")
print("Saved to:", OUTPUT_CSV)
print("Samples:", df.shape[0])
print("Total Features:", df.shape[1] - 3)


🔍 Extracting → TCGA-B0-4712/12-09-1991-NA-CT ABDOMEN WITH CONTRA-96104/2 HELICAL 5s_2_std.nii.gz


parameter force2D must be set to True to enable shape2D extraction
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calc

🔍 Extracting → TCGA-B0-4821/10-28-1984-NA-ABDOMEN WWO-67734/3 Unnamed Series - acquisitionNumber 2_std.nii.gz


parameter force2D must be set to True to enable shape2D extraction
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calc

🔍 Extracting → TCGA-B0-5085/06-18-1987-NA-e1 AP-18854/3 POST CONTRAST_std.nii.gz


...

🔍 Extracting → TCGA-CZ-4857/11-01-1996-NA-ABD WWO CONT-24147/9 MR Ax T1 FSPGR FS POST DYN - 3 frames Volume Sequence by AcquisitionTime 0_std.nii.gz


parameter force2D must be set to True to enable shape2D extraction
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calc

🔍 Extracting → TCGA-CZ-5989/11-21-1997-NA-ABD WWO CONT-16700/11 MR Ax FAME BH POST (X3) - 3 frames Volume Sequence by AcquisitionTime 0_std.nii.gz


parameter force2D must be set to True to enable shape2D extraction
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calc


 RADIOMICS EXTRACTION COMPLETED
Saved to: /kaggle/working/radiomics_results/radiomics_features.csv
Samples: 42
Total Features: 1595


In [4]:
import pandas as pd

df = pd.read_csv("/kaggle/working/radiomics_results/radiomics_features.csv")

# Buang metadata (yang tersedia di CSV)
feature_df = df.drop(columns=["patient_id", "protocol", "image_file"])

feature_classes = {
    "firstorder": 0,
    "shape": 0,
    "shape2D": 0,
    "glcm": 0,
    "glrlm": 0,
    "glszm": 0,
    "gldm": 0,
    "ngtdm": 0
}

# Hitung per kelas
for col in feature_df.columns:
    for cls in feature_classes:
        if f"_{cls}_" in col.lower():   # lower untuk amannya
            feature_classes[cls] += 1

# Tambah total
feature_classes["TOTAL"] = sum(feature_classes.values())

feature_classes


{'firstorder': 306,
 'shape': 14,
 'shape2D': 0,
 'glcm': 408,
 'glrlm': 272,
 'glszm': 272,
 'gldm': 238,
 'ngtdm': 85,
 'TOTAL': 1595}

In [5]:
df.head(5)

,patient_id,protocol,image_file,original_shape_Elongation,original_shape_Flatness,original_shape_LeastAxisLength,original_shape_MajorAxisLength,original_shape_Maximum2DDiameterColumn,original_shape_Maximum2DDiameterRow,original_shape_Maximum2DDiameterSlice,...,gradient_glszm_SmallAreaHighGrayLevelEmphasis,gradient_glszm_SmallAreaLowGrayLevelEmphasis,gradient_glszm_ZoneEntropy,gradient_glszm_ZonePercentage,gradient_glszm_ZoneVariance,gradient_ngtdm_Busyness,gradient_ngtdm_Coarseness,gradient_ngtdm_Complexity,gradient_ngtdm_Contrast,gradient_ngtdm_Strength
0,TCGA-B0-4712,12-09-1991-NA-CT ABDOMEN WITH CONTRA-96104,2 HELICAL 5s_2_std.nii.gz,0.870300,0.798976,50.319432,62.979923,71.589105,75.604233,63.600314,...,1.114643,0.068473,4.683629,0.002623,4.042581e+07,663.772007,0.000257,0.582355,0.000692,0.003382
1,TCGA-B0-4821,10-28-1984-NA-ABDOMEN WWO-67734,3 Unnamed Series - acquisitionNumber 2_std.nii.gz,0.805289,0.694313,72.203311,103.992416,123.555655,122.539789,99.624294,...,1.351292,0.064710,4.986135,0.003717,8.385392e+07,3417.464397,0.000037,2.141977,0.002058,0.000993
2,TCGA-B0-5085,06-18-1987-NA-e1 AP-18854,3 POST CONTRAST_std.nii.gz,0.744143,0.529304,41.979143,79.310087,83.360662,68.942005,86.833173,...,7.514385,0.138558,5.409197,0.009386,4.167307e+06,757.718340,0.000060,25.121919,0.013012,0.005205
3,TCGA-B0-5099,12-29-1984-NA-CT ABDOMEN-49423,3 AXIAL SCANS_std.nii.gz,0.820893,0.643006,51.700880,80.405020,85.586214,89.106678,93.941471,...,1.637175,0.045730,6.172004,0.002542,3.022364e+07,1377.079132,0.000056,9.226622,0.005485,0.003739
4,TCGA-B0-5106,04-11-1987-NA-NA-14123,2 RENALS-T1_AXIAL_BREATHHOLD_std.nii.gz,0.859735,0.725512,31.673953,43.657388,50.328918,47.413078,46.957428,...,30.797630,0.069099,6.774477,0.028442,2.226281e+05,45.530705,0.000256,193.460593,0.017302,0.151750


In [6]:
# ============================================================
# DATA MERGING: RADIOMICS + CLINICAL + DIGEST (TCGA-KIRC)
# ============================================================

import pandas as pd
import numpy as np

# ============================================================
# 0. LOAD DATA
# ============================================================
radiomics_path = "/kaggle/working/radiomics_results/radiomics_features.csv"
clinical_path  = "/kaggle/input/metadata/TCGA-KIRC.clinical.tsv"
digest_path    = "/kaggle/input/metadata/TCIA_TCGA-KIRC_09-16-2015-nbia-digest.xlsx"

df_radiomics = pd.read_csv(radiomics_path)
df_clinical  = pd.read_csv(clinical_path, sep="\t")
df_digest    = pd.read_excel(digest_path)

print("Loaded:")
print(" Radiomics :", df_radiomics.shape)
print(" Clinical  :", df_clinical.shape)
print(" Digest    :", df_digest.shape)

# ============================================================
# 1. PREPARE CLINICAL DATA
# ============================================================
cols_needed_clinical = [
    "submitter_id",
    "ajcc_pathologic_stage.diagnoses",
    "ajcc_pathologic_t.diagnoses",
    "ajcc_pathologic_n.diagnoses",
    "ajcc_pathologic_m.diagnoses"
]

df_clinical_grouped = (
    df_clinical[cols_needed_clinical]
    .groupby("submitter_id")
    .agg(lambda x: ", ".join(x.dropna().unique().astype(str)))
    .reset_index()
    .rename(columns={
        "submitter_id": "patient_id_klinis",
        "ajcc_pathologic_stage.diagnoses": "Stage",
        "ajcc_pathologic_t.diagnoses": "T_Staging",
        "ajcc_pathologic_n.diagnoses": "N_Staging",
        "ajcc_pathologic_m.diagnoses": "M_Staging"
    })
)

df_clinical_final = df_clinical_grouped[
    ~df_clinical_grouped["Stage"].str.contains(
        "Not Reported|Not Available", na=False
    )
].copy()

print("Clinical after filtering:", df_clinical_final.shape)

# ============================================================
# 2. PREPARE RADIOMICS METADATA
# ============================================================
df_radiomics["patient_id_short"] = (
    df_radiomics["patient_id"]
    .astype(str)
    .apply(lambda x: "-".join(x.split("-")[:3]))
)

def clean_protocol_date(protocol):
    try:
        date_part = protocol.split("-NA-")[0]
        m, d, y = date_part.split("-")
        return f"{y}-{m}-{d}"
    except:
        return np.nan

df_radiomics["Series Date_cleaned"] = df_radiomics["protocol"].apply(clean_protocol_date)

# ============================================================
# 3. PREPARE DIGEST DATA (FIX DUPLICATE)
# ============================================================
df_digest["Series Date_cleaned"] = (
    df_digest["Series Date"]
    .astype(str)
    .str.split(" ")
    .str[0]
)

# 🔥 PENTING: buang duplikat digest
df_digest_unique = (
    df_digest[["Patient ID", "Series Date_cleaned"]]
    .drop_duplicates()
    .rename(columns={
        "Patient ID": "patient_id_klinis"
    })
)

print("Digest after dedup:", df_digest_unique.shape)

# ============================================================
# 4. MERGE: RADIOMICS + CLINICAL
# ============================================================
df_merged = pd.merge(
    df_radiomics,
    df_clinical_final,
    left_on="patient_id_short",
    right_on="patient_id_klinis",
    how="inner"   # ❗ ganti inner biar target valid
)

print("After clinical merge:", df_merged.shape)

# ============================================================
# 5. MERGE: + DIGEST (STRICT DATE MATCH)
# ============================================================
df_merged_final = pd.merge(
    df_merged,
    df_digest_unique,
    left_on=["patient_id_klinis", "Series Date_cleaned"],
    right_on=["patient_id_klinis", "Series Date_cleaned"],
    how="inner"   # ❗ jangan left
)

print("After digest merge:", df_merged_final.shape)

# ============================================================
# 6. TARGET ENCODING
# ============================================================
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_merged_final["Target"] = le.fit_transform(df_merged_final["Stage"])

df_merged_final = df_merged_final.dropna(subset=["Target"]).reset_index(drop=True)

print("Stage encoding:")
print(dict(zip(le.classes_, le.transform(le.classes_))))

# ============================================================
# 7. REORDER COLUMNS (METADATA FIRST)
# ============================================================

metadata_cols = [
    # --- Radiomics metadata ---
    "patient_id",
    "patient_id_short",
    "protocol",
    "image_file",
    "Series Date_cleaned",

    # --- Clinical ---
    "patient_id_klinis",
    "Stage",
    "T_Staging",
    "N_Staging",
    "M_Staging",

    # --- Target ---
    "Target"
]

# Ambil radiomics feature columns saja
radiomics_cols = [
    c for c in df_merged_final.columns
    if c not in metadata_cols
    and not c.startswith("diagnostics")
    and df_merged_final[c].dtype != object
]

# Reorder dataframe
df_merged_final = df_merged_final[
    metadata_cols + radiomics_cols
]

print("Metadata columns:", len(metadata_cols))
print("Radiomics features:", len(radiomics_cols))
print("Total columns:", df_merged_final.shape[1])

# ============================================================
# 8. FINAL CHECK
# ============================================================
print("\nFinal column order preview:")
display(df_merged_final.iloc[:, :15].head())

# ============================================================
# 9. SAVE MERGED DATA
# ============================================================

output_path = "/kaggle/working/radiomics_results/radiomics_clinical_digest_merged.csv"

df_merged_final.to_csv(output_path, index=False)

print("\n✅ File saved successfully!")
print("Path:", output_path)
print("Final shape:", df_merged_final.shape)


Loaded:
 Radiomics : (42, 1598)
 Clinical  : (950, 90)
 Digest    : (2654, 36)
Clinical after filtering: (537, 5)
Digest after dedup: (444, 2)
After clinical merge: (42, 1605)
After digest merge: (42, 1605)
Stage encoding:
{'Stage I': np.int64(0), 'Stage II': np.int64(1), 'Stage III': np.int64(2), 'Stage IV': np.int64(3)}
Metadata columns: 11
Radiomics features: 1595
Total columns: 1606

Final column order preview:


,patient_id,patient_id_short,protocol,image_file,Series Date_cleaned,patient_id_klinis,Stage,T_Staging,N_Staging,M_Staging,Target,original_shape_Elongation,original_shape_Flatness,original_shape_LeastAxisLength,original_shape_MajorAxisLength
0,TCGA-B0-4712,TCGA-B0-4712,12-09-1991-NA-CT ABDOMEN WITH CONTRA-96104,2 HELICAL 5s_2_std.nii.gz,1991-12-09,TCGA-B0-4712,Stage IV,T3a,NX,M1,3,0.870300,0.798976,50.319432,62.979923
1,TCGA-B0-4821,TCGA-B0-4821,10-28-1984-NA-ABDOMEN WWO-67734,3 Unnamed Series - acquisitionNumber 2_std.nii.gz,1984-10-28,TCGA-B0-4821,Stage III,T3b,N0,M0,2,0.805289,0.694313,72.203311,103.992416
2,TCGA-B0-5085,TCGA-B0-5085,06-18-1987-NA-e1 AP-18854,3 POST CONTRAST_std.nii.gz,1987-06-18,TCGA-B0-5085,Stage III,T3a,N0,M0,2,0.744143,0.529304,41.979143,79.310087
3,TCGA-B0-5099,TCGA-B0-5099,12-29-1984-NA-CT ABDOMEN-49423,3 AXIAL SCANS_std.nii.gz,1984-12-29,TCGA-B0-5099,Stage III,T3b,NX,M0,2,0.820893,0.643006,51.700880,80.405020
4,TCGA-B0-5106,TCGA-B0-5106,04-11-1987-NA-NA-14123,2 RENALS-T1_AXIAL_BREATHHOLD_std.nii.gz,1987-04-11,TCGA-B0-5106,Stage I,T1a,N0,M0,0,0.859735,0.725512,31.673953,43.657388



✅ File saved successfully!
Path: /kaggle/working/radiomics_results/radiomics_clinical_digest_merged.csv
Final shape: (42, 1606)


In [7]:
# ============================================================
# mRMRe + CORRELATION THRESHOLD (85%)
# FIXED FOR PYMRMR - KAGGLE
# ============================================================

import pandas as pd
import numpy as np
import pymrmr
from sklearn.preprocessing import KBinsDiscretizer, LabelEncoder

# ============================================================
# 1. LOAD DATA
# ============================================================
RADIOMICS_PATH = "/kaggle/working/radiomics_results/radiomics_clinical_digest_merged.csv"
OUTPUT_MRMR_PATH = "/kaggle/working/selected_radiomics_features_mrmre.tsv"

df = pd.read_csv(RADIOMICS_PATH)

# ============================================================
# 2. TARGET ENCODING
# ============================================================
le = LabelEncoder()
df["Target"] = le.fit_transform(df["Stage"])
df = df.dropna(subset=["Target"]).reset_index(drop=True)

print("Stage encoding:", dict(zip(le.classes_, le.transform(le.classes_))))

# ============================================================
# 3. AMBIL FITUR RADIOMICS SAJA
# ============================================================
metadata_cols = [
    "patient_id", "patient_id_short", "protocol",
    "image_file",
    "Stage", "T_Staging", "N_Staging", "M_Staging",
    "Target", "Series Date_cleaned"
]

radiomics_cols = [
    c for c in df.columns
    if c not in metadata_cols
    and not c.startswith("diagnostics")
    and df[c].dtype != object
]

X = df[radiomics_cols]

print("Initial radiomics features:", X.shape[1])

# ============================================================
# 4. FILTER KORELASI
# ============================================================
CORRELATION_THRESHOLD = 0.85

corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] > CORRELATION_THRESHOLD)]

X_filtered = X.drop(columns=to_drop)

print(f"Removed by correlation filter: {len(to_drop)}")
print(f"Remaining features: {X_filtered.shape[1]}")

# ============================================================
# 5. DISCRETIZATION (WAJIB UNTUK pymrmr)
# ============================================================
discretizer = KBinsDiscretizer(
    n_bins=10,
    encode="ordinal",
    strategy="quantile"
)

X_binned = discretizer.fit_transform(X_filtered)
X_binned = pd.DataFrame(
    X_binned,
    columns=X_filtered.columns
).astype(int)

# ============================================================
# 6. FORMAT KHUSUS pymrmr
#    - Target HARUS kolom pertama
#    - Semua INT
# ============================================================
df_mrmre = pd.concat(
    [df["Target"].astype(int), X_binned],
    axis=1
)

# ============================================================
# 7. mRMRe FEATURE SELECTION
# ============================================================
N_FEATURES_MRMRE = 20

selected_features = pymrmr.mRMR(
    df_mrmre,
    "MIQ",        # Mutual Information Quotient
    N_FEATURES_MRMRE
)

print("\nSelected features by mRMRe:")
for f in selected_features:
    print(" -", f)

# ============================================================
# 8. SAVE FINAL DATASET
# ============================================================
final_cols = [
    "patient_id_short",
    "Stage", "T_Staging", "N_Staging", "M_Staging",
    "Target"
] + selected_features

df_final = df[final_cols]

df_final.to_csv(OUTPUT_MRMR_PATH, sep="\t", index=False)

print("\n======================================")
print(" mRMRe + Correlation Filtering DONE")
print("======================================")
print("Saved file :", OUTPUT_MRMR_PATH)
print("Samples    :", df_final.shape[0])
print("Features   :", len(selected_features))
print("======================================")


Stage encoding: {'Stage I': np.int64(0), 'Stage II': np.int64(1), 'Stage III': np.int64(2), 'Stage IV': np.int64(3)}
Initial radiomics features: 1595
Removed by correlation filter: 1421
Remaining features: 174


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 100 are removed. Consider decreasing the number of bins.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 101 are removed. Consider decreasing the number of bins.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 102 are removed. Consider decreasing the number of bins.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 103 are removed. Consider decreasing the number of bins.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_discretizatio


Selected features by mRMRe:
 - original_shape_LeastAxisLength
 - square_firstorder_Minimum
 - log-sigma-2-0-mm-3D_glszm_SizeZoneNonUniformityNormalized
 - wavelet-LLH_glszm_GrayLevelNonUniformity
 - original_glszm_ZoneVariance
 - wavelet-LHH_firstorder_Entropy
 - wavelet-LHL_glcm_Correlation
 - log-sigma-2-0-mm-3D_glszm_LargeAreaHighGrayLevelEmphasis
 - original_firstorder_Minimum
 - wavelet-HLL_glcm_InverseVariance
 - wavelet-LLH_glszm_LargeAreaEmphasis
 - square_glcm_Correlation
 - wavelet-LHH_firstorder_Median
 - wavelet-LLH_glcm_Idmn
 - wavelet-LHH_glcm_ClusterShade
 - wavelet-LLH_ngtdm_Coarseness
 - wavelet-HHH_gldm_DependenceEntropy
 - wavelet-HHL_firstorder_Skewness
 - logarithm_firstorder_Energy
 - wavelet-HLL_firstorder_Skewness

 mRMRe + Correlation Filtering DONE
Saved file : /kaggle/working/selected_radiomics_features_mrmre.tsv
Samples    : 42
Features   : 20


In [8]:
# ============================================================
# GENETIC ALGORITHM FEATURE SELECTION (DEAP) - KAGGLE
# Wrapper Method (MULTICLASS SAFE)
# Output: selected_radiomics_features_ga_deap.tsv
# ============================================================

import pandas as pd
import numpy as np
import random
import warnings

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.exceptions import UndefinedMetricWarning

from deap import base, creator, tools, algorithms

# ============================================================
# 0. SUPPRESS ALL ANNOYING WARNINGS (PENTING)
# ============================================================
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

# ============================================================
# 1. LOAD MERGED DATA
# ============================================================
DATA_PATH = "/kaggle/working/radiomics_results/radiomics_clinical_digest_merged.csv"
OUTPUT_GA_PATH = "/kaggle/working/selected_radiomics_features_ga_deap.tsv"

df = pd.read_csv(DATA_PATH)
print("Data loaded:", df.shape)

# ============================================================
# 2. TARGET ENCODING (MULTICLASS)
# ============================================================
le = LabelEncoder()
df["Target"] = le.fit_transform(df["Stage"])
df = df.dropna(subset=["Target"]).reset_index(drop=True)

print("Stage encoding:", dict(zip(le.classes_, le.transform(le.classes_))))

# ============================================================
# 3. FEATURE MATRIX (RADIOMICS ONLY)
# ============================================================
metadata_cols = [
    "patient_id", "patient_id_short", "protocol",
    "image_file",
    "Stage", "T_Staging", "N_Staging", "M_Staging",
    "Target", "Series Date_cleaned"
]

radiomics_cols = [
    c for c in df.columns
    if c not in metadata_cols
    and not c.startswith("diagnostics")
    and df[c].dtype != object
]

X = df[radiomics_cols]
y = df["Target"]

print("Total radiomics features:", X.shape[1])

# ============================================================
# 4. FILTER KORELASI
# ============================================================
CORRELATION_THRESHOLD = 0.85

corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] > CORRELATION_THRESHOLD)]

X_filtered = X.drop(columns=to_drop)

print(f"Removed by correlation filter: {len(to_drop)}")
print(f"Remaining features: {X_filtered.shape[1]}")

# ============================================================
# 5. SCALING
# ============================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filtered)
X_scaled = pd.DataFrame(X_scaled, columns=X_filtered.columns)

# ============================================================
# 6. GENETIC ALGORITHM SETUP
# ============================================================
random.seed(42)
np.random.seed(42)

N_FEATURES = X_scaled.shape[1]
POP_SIZE = 50
N_GEN = 20
CXPB = 0.7
MUTPB = 0.2

estimator = LogisticRegression(
    solver="liblinear",
    max_iter=1000,
    random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ============================================================
# 7. FITNESS FUNCTION (ROC AUC OVR — FIXED)
# ============================================================
def eval_individual(individual):
    if sum(individual) == 0:
        return (0.0,)

    idx = [i for i, bit in enumerate(individual) if bit == 1]
    X_sel = X_scaled.iloc[:, idx]

    try:
        scores = cross_val_score(
            estimator,
            X_sel,
            y,
            cv=cv,
            scoring="roc_auc_ovr",  # 🔥 FIX MULTICLASS
            n_jobs=-1
        )
        if np.isnan(scores).any():
            return (0.0,)
        return (scores.mean(),)
    except:
        return (0.0,)

# ============================================================
# 8. DEAP CONFIGURATION (SAFE FOR RE-RUN)
# ============================================================
if "FitnessMax" not in creator.__dict__:
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
if "Individual" not in creator.__dict__:
    creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, N_FEATURES)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", eval_individual)
toolbox.register("mate", tools.cxUniform, indpb=0.5)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

# ============================================================
# 9. RUN GENETIC ALGORITHM
# ============================================================
population = toolbox.population(n=POP_SIZE)
halloffame = tools.HallOfFame(1)

algorithms.eaSimple(
    population,
    toolbox,
    cxpb=CXPB,
    mutpb=MUTPB,
    ngen=N_GEN,
    halloffame=halloffame,
    verbose=False
)

best_individual = halloffame[0]
best_score = best_individual.fitness.values[0]

selected_indices = [i for i, bit in enumerate(best_individual) if bit == 1]
selected_features = list(X_filtered.columns[selected_indices])

print("\nSelected features by GA:", len(selected_features))
print("Best ROC AUC (OVR):", round(best_score, 4))

# ============================================================
# 10. SAVE FINAL DATASET
# ============================================================
final_cols = [
    "patient_id_short",
    "Stage", "T_Staging", "N_Staging", "M_Staging",
    "Target"
] + selected_features

df_final = df[final_cols]
df_final.to_csv(OUTPUT_GA_PATH, sep="\t", index=False)

print("\n======================================")
print(" GENETIC ALGORITHM FEATURE SELECTION DONE")
print("======================================")
print("Saved file :", OUTPUT_GA_PATH)
print("Samples    :", df_final.shape[0])
print("Features   :", len(selected_features))
print("ROC AUC    :", round(best_score, 4))
print("======================================")


Data loaded: (42, 1606)
Stage encoding: {'Stage I': np.int64(0), 'Stage II': np.int64(1), 'Stage III': np.int64(2), 'Stage IV': np.int64(3)}
Total radiomics features: 1595
Removed by correlation filter: 1421
Remaining features: 174

Selected features by GA: 63
Best ROC AUC (OVR): 0.9185

 GENETIC ALGORITHM FEATURE SELECTION DONE
Saved file : /kaggle/working/selected_radiomics_features_ga_deap.tsv
Samples    : 42
Features   : 63
ROC AUC    : 0.9185


In [9]:
# ============================================================
# L1 LASSO FEATURE SELECTION (EMBEDDED METHOD) - KAGGLE
# Multiclass Logistic Regression (OVR)
# Output: selected_radiomics_features_l1_lasso.tsv
# ============================================================

import pandas as pd
import numpy as np
import warnings
import os

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold
from sklearn.exceptions import ConvergenceWarning

# ============================================================
# 0. SUPPRESS WARNINGS (BIAR KAGGLE NGGAK NANGIS)
# ============================================================
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# ============================================================
# 1. LOAD DATA (MERGED)
# ============================================================
DATA_PATH = "/kaggle/working/radiomics_results/radiomics_clinical_digest_merged.csv"
OUTPUT_PATH = "/kaggle/working/selected_radiomics_features_l1_lasso.tsv"

df = pd.read_csv(DATA_PATH)
print("Data loaded:", df.shape)

# ============================================================
# 2. TARGET ENCODING (STAGE)
# ============================================================
le = LabelEncoder()
df["Target"] = le.fit_transform(df["Stage"])
df = df.dropna(subset=["Target"]).reset_index(drop=True)

print("Stage encoding:", dict(zip(le.classes_, le.transform(le.classes_))))

# ============================================================
# 3. RADIOMICS FEATURE MATRIX
# ============================================================
metadata_cols = [
    "patient_id", "patient_id_short", "protocol",
    "image_file",
    "Stage", "T_Staging", "N_Staging", "M_Staging",
    "Target", "Series Date_cleaned"
]

radiomics_cols = [
    c for c in df.columns
    if c not in metadata_cols
    and not c.startswith("diagnostics")
    and df[c].dtype != object
]

X = df[radiomics_cols]
y = df["Target"]

print("Total radiomics features:", X.shape[1])

# ============================================================
# 4. FILTER KORELASI (WAJIB)
# ============================================================
CORRELATION_THRESHOLD = 0.85

corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] > CORRELATION_THRESHOLD)]

X_filtered = X.drop(columns=to_drop)

print(f"Removed by correlation filter: {len(to_drop)}")
print(f"Remaining features: {X_filtered.shape[1]}")

# ============================================================
# 5. SCALING
# ============================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filtered)

# ============================================================
# 6. L1 LASSO (LOGISTIC REGRESSION – EMBEDDED)
# ============================================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lasso_l1 = LogisticRegressionCV(
    Cs=10,
    cv=cv,
    penalty="l1",
    solver="saga",
    multi_class="ovr",
    scoring="roc_auc_ovr",
    max_iter=5000,
    n_jobs=-1,
    random_state=42
)

lasso_l1.fit(X_scaled, y)

# ============================================================
# 7. FEATURE SELECTION
# ============================================================
coef = np.mean(np.abs(lasso_l1.coef_), axis=0)
selected_mask = coef > 0
selected_features = X_filtered.columns[selected_mask].tolist()

print("\nSelected features by L1 LASSO:", len(selected_features))
print("Best C (regularization strength):", lasso_l1.C_[0])

# ============================================================
# 8. SAVE FINAL DATASET
# ============================================================
final_cols = [
    "patient_id_short",
    "Stage", "T_Staging", "N_Staging", "M_Staging",
    "Target"
] + selected_features

df_final = df[final_cols]
df_final.to_csv(OUTPUT_PATH, sep="\t", index=False)

print("\n======================================")
print(" L1 LASSO FEATURE SELECTION DONE")
print("======================================")
print("Saved file :", OUTPUT_PATH)
print("Samples    :", df_final.shape[0])
print("Features   :", len(selected_features))
print("======================================")


Data loaded: (42, 1606)
Stage encoding: {'Stage I': np.int64(0), 'Stage II': np.int64(1), 'Stage III': np.int64(2), 'Stage IV': np.int64(3)}
Total radiomics features: 1595
Removed by correlation filter: 1421
Remaining features: 174


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1917: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegressionCV(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(



Selected features by L1 LASSO: 49
Best C (regularization strength): 0.3593813663804626

 L1 LASSO FEATURE SELECTION DONE
Saved file : /kaggle/working/selected_radiomics_features_l1_lasso.tsv
Samples    : 42
Features   : 49
